In [ ]:
%pip install requests python-dotenv
%pip install geopy azure-cognitiveservices-speech

In [3]:
# 🔊 FR-005-SimplifiedTTS: 개선된 음성 안내 서비스
# 핵심 개선사항: .env 기반 경로 관리 + 프로젝트 루트 자동 감지

import pandas as pd
import numpy as np
import os
import math
from datetime import datetime, timedelta
from dotenv import load_dotenv
import warnings
from pathlib import Path

# Azure Speech Services
try:
    import azure.cognitiveservices.speech as speechsdk
    from azure.cognitiveservices.speech.audio import AudioOutputConfig
    AZURE_AVAILABLE = True
    print("✅ Azure Speech SDK 로드 완료")
except ImportError:
    AZURE_AVAILABLE = False
    print("⚠️ Azure Speech SDK가 설치되지 않았습니다. 텍스트만 출력됩니다.")

warnings.filterwarnings('ignore')

# =============================================================================
# 0. 프로젝트 경로 관리 클래스
# =============================================================================

class ProjectPathManager:
    """프로젝트 루트 기준 경로 관리"""
    
    def __init__(self, manual_root=None):
        if manual_root:
            self.project_root = Path(manual_root).resolve()
            print(f"📁 수동 설정된 프로젝트 루트: {self.project_root}")
        else:
            self.project_root = self._find_project_root()
        self._load_env_config()
        
    def _find_project_root(self):
        """프로젝트 루트 디렉토리 자동 감지"""
        current_path = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()
        
        print(f"🔍 현재 경로: {current_path}")
        
        # 1. 프로젝트 특정 마커들을 찾아서 루트 결정
        # backend와 data 폴더가 모두 있는 디렉토리를 찾기
        for parent in [current_path] + list(current_path.parents):
            print(f"   검사 중: {parent}")
            
            # Sesac_5_perror 같은 프로젝트 이름이 포함된 경우 우선
            if any(name in parent.name.lower() for name in ['sesac', 'perror', 'project']):
                if (parent / 'backend').exists() and (parent / 'data').exists():
                    print(f"📁 프로젝트 루트 감지 (이름+구조): {parent}")
                    return parent
            
            # backend와 data 폴더가 모두 있는 경우
            if (parent / 'backend').exists() and (parent / 'data').exists():
                print(f"📁 프로젝트 루트 감지 (구조): {parent}")
                return parent
            
            # .env 파일이 있으면서 backend 또는 data 폴더가 있는 경우
            if (parent / '.env').exists():
                if (parent / 'backend').exists() or (parent / 'data').exists():
                    print(f"📁 프로젝트 루트 감지 (.env+구조): {parent}")
                    return parent
        
        # 2. 현재 위치가 backend/FR-005 같은 하위 폴더인 경우 상위로 이동
        if current_path.name == 'FR-005' and current_path.parent.name == 'backend':
            potential_root = current_path.parent.parent
            print(f"📁 프로젝트 루트 추정 (backend/FR-005 패턴): {potential_root}")
            return potential_root
        elif current_path.name == 'backend':
            potential_root = current_path.parent
            print(f"📁 프로젝트 루트 추정 (backend 패턴): {potential_root}")
            return potential_root
        
        # 3. 기본 마커 파일들로 검색
        standard_markers = ['.env', 'requirements.txt', '.git', 'package.json']
        for parent in [current_path] + list(current_path.parents):
            if any((parent / marker).exists() for marker in standard_markers):
                print(f"📁 프로젝트 루트 감지 (마커): {parent}")
                return parent
        
        # 4. 최종 대안: 현재 경로의 적절한 상위
        fallback_root = current_path
        if current_path.name in ['FR-005', 'backend']:
            fallback_root = current_path.parent
            if fallback_root.name == 'backend':
                fallback_root = fallback_root.parent
        
        print(f"📁 프로젝트 루트 최종 추정: {fallback_root}")
        return fallback_root
    
    def _load_env_config(self):
        """환경 변수 로드"""
        env_path = self.project_root / '.env'
        if env_path.exists():
            load_dotenv(env_path)
            print(f"✅ .env 파일 로드: {env_path}")
        else:
            print(f"⚠️ .env 파일 없음: {env_path}")

    
    def get_construction_data_paths(self):
        """공사 데이터 파일 경로 목록 반환"""
        paths = []
        
        # 1. 주 경로 (.env에서)
        main_path = os.getenv('CONSTRUCTION_DATA_PATH')
        if main_path:
            paths.append(self.project_root / main_path)
        
        # 2. 대체 경로들 (.env에서)
        fallback_paths = os.getenv('CONSTRUCTION_DATA_FALLBACK_PATHS', '')
        if fallback_paths:
            for path in fallback_paths.split(','):
                paths.append(self.project_root / path.strip())
        
        # 3. 하드코딩된 기본 경로들
        default_paths = [
            'data/processed/road_excavation_processed_20250623_143840.csv',
            'data/csv/서울시 도로굴착 공사 현황.csv',
            'data/csv/서울시 신설공사 추진 현황.csv',
            'backend/FR-005/sample_data.csv'  # 샘플 데이터 경로
        ]
        
        for path in default_paths:
            full_path = self.project_root / path
            if full_path not in paths:
                paths.append(full_path)
        
        return paths
    
    
    def get_absolute_path(self, relative_path):
        """상대 경로를 절대 경로로 변환"""
        return self.project_root / relative_path
    
    def list_available_data_files(self):
        """사용 가능한 데이터 파일 목록 출력"""
        print(f"\n📂 사용 가능한 데이터 파일 검색...")
        print(f"   프로젝트 루트: {self.project_root}")
        
        paths = self.get_construction_data_paths()
        available_files = []
        
        for path in paths:
            if path.exists():
                file_size = path.stat().st_size
                print(f"   ✅ {path.relative_to(self.project_root)} ({file_size:,} bytes)")
                available_files.append(path)
            else:
                print(f"   ❌ {path.relative_to(self.project_root)}")
        
        if not available_files:
            print(f"   📋 data 폴더 구조:")
            data_dir = self.project_root / 'data'
            if data_dir.exists():
                for item in data_dir.rglob('*.csv'):
                    print(f"      📄 {item.relative_to(self.project_root)}")
            else:
                print(f"      ❌ data 폴더가 존재하지 않습니다.")
        
        return available_files

# 전역 경로 관리자 초기화
def init_path_manager(manual_root=None):
    """경로 관리자 초기화 (수동 루트 설정 가능)"""
    global path_manager
    path_manager = ProjectPathManager(manual_root)
    return path_manager

def fix_project_root():
    """프로젝트 루트 경로 수정"""
    global path_manager
    
    print(f"🔧 프로젝트 루트 경로 수정")
    print(f"현재 감지된 루트: {path_manager.project_root}")
    
    # 현재 경로에서 Sesac_5_perror 찾기
    current = Path.cwd()
    print(f"현재 작업 디렉토리: {current}")
    
    # 상위 디렉토리들 중에서 Sesac_5_perror 찾기
    for parent in [current] + list(current.parents):
        if 'sesac' in parent.name.lower() or 'perror' in parent.name.lower():
            print(f"✅ Sesac_5_perror 디렉토리 발견: {parent}")
            
            # 수동으로 경로 관리자 재초기화
            path_manager = ProjectPathManager(manual_root=parent)
            
            # 확인
            print(f"📁 수정된 프로젝트 루트: {path_manager.project_root}")
            
            # data 폴더 확인
            data_dir = path_manager.project_root / 'data'
            backend_dir = path_manager.project_root / 'backend'
            
            print(f"📂 폴더 구조 확인:")
            print(f"   data 폴더: {'✅ 존재' if data_dir.exists() else '❌ 없음'} ({data_dir})")
            print(f"   backend 폴더: {'✅ 존재' if backend_dir.exists() else '❌ 없음'} ({backend_dir})")
            
            if data_dir.exists():
                print(f"📄 data 폴더 내용:")
                for item in data_dir.rglob('*'):
                    if item.is_file():
                        print(f"      📄 {item.relative_to(path_manager.project_root)}")
            
            return path_manager
    
    print(f"❌ Sesac_5_perror 디렉토리를 찾을 수 없습니다.")
    print(f"💡 수동으로 설정하려면:")
    print(f"   path_manager = init_path_manager('정확한/경로/Sesac_5_perror')")
    
    return None

# 초기 경로 관리자 설정 (함수 정의 이후)
try:
    path_manager = ProjectPathManager()
except Exception as e:
    print(f"⚠️ 초기 경로 관리자 설정 실패: {e}")
    print(f"💡 quick_fix_and_test() 또는 init_path_manager()를 사용하세요.")

print("🔊 FR-005-SimplifiedTTS: 개선된 음성 안내 서비스")
print("🎯 프로젝트 루트 기준 경로 관리 + 음성 안내")
print("=" * 60)

# =============================================================================
# 1. 거리 계산 함수 (기존과 동일)
# =============================================================================

def calculate_distance(lat1, lng1, lat2, lng2):
    """두 좌표 간 거리 계산 (미터) - Haversine 공식"""
    R = 6371000  # 지구 반지름 (미터)
    
    lat1_rad = math.radians(lat1)
    lng1_rad = math.radians(lng1)
    lat2_rad = math.radians(lat2)
    lng2_rad = math.radians(lng2)
    
    dlat = lat2_rad - lat1_rad
    dlng = lng2_rad - lng1_rad
    
    a = (math.sin(dlat/2)**2 + 
         math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlng/2)**2)
    c = 2 * math.asin(math.sqrt(a))
    
    return R * c

# =============================================================================
# 2. 개선된 공사현황 데이터 로드 (경로 관리 포함)
# =============================================================================

def load_construction_data_smart():
    """
    스마트 공사현황 데이터 로드 (.env 기반 경로 관리)
    
    Returns:
        pd.DataFrame: 로드된 공사현황 데이터
    """
    print(f"🔍 스마트 데이터 로드 시작...")
    
    # 사용 가능한 파일 목록 확인
    available_files = path_manager.list_available_data_files()
    
    if not available_files:
        print(f"❌ 사용 가능한 데이터 파일이 없습니다. 샘플 데이터를 생성합니다.")
        return create_sample_construction_data()
    
    # 첫 번째 사용 가능한 파일 로드
    for file_path in available_files:
        try:
            print(f"\n📂 데이터 로드 시도: {file_path.name}")
            
            # CSV 파일 로드
            df = pd.read_csv(file_path, encoding='utf-8-sig')
            
            print(f"✅ 파일 로드 성공: {len(df)}건")
            print(f"📋 컬럼: {list(df.columns)}")
            
            # 데이터 전처리
            processed_df = preprocess_construction_data(df)
            
            if len(processed_df) > 0:
                print(f"🎯 전처리 완료: {len(processed_df)}건 (유효한 진행중 공사)")
                return processed_df
            else:
                print(f"⚠️ 전처리 후 유효한 데이터가 없습니다. 다음 파일을 시도합니다.")
                continue
                
        except Exception as e:
            print(f"❌ 파일 로드 실패: {str(e)}")
            continue
    
    # 모든 파일 로드 실패 시 샘플 데이터
    print(f"📊 모든 파일 로드 실패. 샘플 데이터로 대체합니다.")
    return create_sample_construction_data()

def preprocess_construction_data(df):
    """공사현황 데이터 전처리"""
    try:
        # 날짜 컬럼 파싱
        date_columns = ['착공일', '준공일', '시작일', '종료일', '시작날짜', '종료날짜']
        for col in date_columns:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
        
        # 진행중인 공사 필터링
        today = datetime.now()
        
        # 다양한 컬럼명 대응
        status_columns = ['공사상태', '상태', '진행상태', 'status']
        ongoing_keywords = ['진행중', '진행', '공사중', '시공중', 'ongoing', 'active']
        
        ongoing_mask = pd.Series([False] * len(df))
        
        for col in status_columns:
            if col in df.columns:
                for keyword in ongoing_keywords:
                    ongoing_mask |= df[col].astype(str).str.contains(keyword, case=False, na=False)
        
        # 날짜 기반 필터링 추가
        for start_col in ['착공일', '시작일', '시작날짜']:
            for end_col in ['준공일', '종료일', '종료날짜']:
                if start_col in df.columns and end_col in df.columns:
                    date_mask = (
                        (df[start_col] <= today) & 
                        (df[end_col] >= today)
                    )
                    ongoing_mask |= date_mask
        
        df_ongoing = df[ongoing_mask].copy()
        print(f"🚧 진행중인 공사: {len(df_ongoing)}건")
        
        # 좌표 유효성 검증
        coord_columns = {
            'lat': ['위도', 'latitude', 'lat', 'y', 'Y'],
            'lng': ['경도', 'longitude', 'lng', 'lon', 'x', 'X']
        }
        
        lat_col = None
        lng_col = None
        
        for col in coord_columns['lat']:
            if col in df_ongoing.columns:
                lat_col = col
                break
        
        for col in coord_columns['lng']:
            if col in df_ongoing.columns:
                lng_col = col
                break
        
        if lat_col and lng_col:
            # 서울 지역 좌표 범위로 제한
            valid_coords = (
                df_ongoing[lat_col].notna() & 
                df_ongoing[lng_col].notna() &
                (df_ongoing[lat_col] >= 37.4) & (df_ongoing[lat_col] <= 37.7) &
                (df_ongoing[lng_col] >= 126.8) & (df_ongoing[lng_col] <= 127.2)
            )
            
            df_valid = df_ongoing[valid_coords].copy()
            
            # 표준 컬럼명으로 통일
            if lat_col != '위도':
                df_valid['위도'] = df_valid[lat_col]
            if lng_col != '경도':
                df_valid['경도'] = df_valid[lng_col]
            
            print(f"📍 유효한 좌표: {len(df_valid)}건")
            
            # 좌표상태 추가
            df_valid['좌표상태'] = '실제데이터'
            
            return df_valid
        else:
            print(f"❌ 좌표 컬럼을 찾을 수 없습니다. 사용 가능한 컬럼: {list(df_ongoing.columns)}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"❌ 데이터 전처리 실패: {str(e)}")
        return pd.DataFrame()

def create_sample_construction_data():
    """샘플 공사 데이터 생성 (백업용)"""
    sample_data = {
        '관리코드': ['DEMO-001', 'DEMO-002', 'DEMO-003', 'DEMO-004', 'DEMO-005'],
        '지오코딩주소': [
            '서울시 강남구 역삼동',
            '서울시 강남구 선릉역',
            '서울시 강남구 강남역',
            '서울시 서초구 교대역',
            '서울시 서초구 방배동'
        ],
        '착공일': [
            datetime(2024, 11, 1),
            datetime(2024, 10, 15),
            datetime(2024, 12, 1),
            datetime(2024, 11, 20),
            datetime(2024, 10, 1)
        ],
        '준공일': [
            datetime(2025, 2, 28),
            datetime(2025, 1, 30),
            datetime(2025, 3, 15),
            datetime(2025, 1, 15),
            datetime(2024, 12, 31)
        ],
        '공사상태': ['진행중', '진행중', '진행중', '진행중', '진행중'],
        '위도': [37.4979, 37.5044, 37.4979, 37.4951, 37.4816],
        '경도': [127.0276, 127.0373, 127.0276, 127.0176, 126.9976],
        '좌표상태': ['샘플데이터'] * 5
    }
    
    print(f"📊 샘플 데이터 생성: {len(sample_data['관리코드'])}건")
    return pd.DataFrame(sample_data)

def analyze_nearby_constructions(user_location, construction_data, search_radius=500):
    """사용자 주변 공사현황 분석 (기존과 동일)"""
    print(f"🔍 주변 공사 분석 시작...")
    print(f"   사용자 위치: ({user_location['lat']:.6f}, {user_location['lng']:.6f})")
    print(f"   검색 반경: {search_radius}m")
    print(f"   대상 공사: {len(construction_data)}건")
    
    nearby_constructions = []
    
    for idx, construction in construction_data.iterrows():
        # 거리 계산
        distance = calculate_distance(
            user_location['lat'], user_location['lng'],
            construction['위도'], construction['경도']
        )
        
        if distance <= search_radius:
            # 위험도 계산
            if distance <= 100:
                risk_level = '높음'
                urgency = 'immediate'
            elif distance <= 300:
                risk_level = '중간'
                urgency = 'warning'
            else:
                risk_level = '낮음'
                urgency = 'notice'
            
            # 공사 정보 추가
            construction_info = {
                'id': construction.get('관리코드', f'CONST-{idx}'),
                'location': construction.get('지오코딩주소', construction.get('주소', '위치정보없음')),
                'distance': round(distance, 1),
                'risk_level': risk_level,
                'urgency': urgency,
                'end_date': construction.get('준공일', '미정'),
                'coordinate_status': construction.get('좌표상태', '알수없음'),
                'construction_status': construction.get('공사상태', '알수없음')
            }
            
            nearby_constructions.append(construction_info)
    
    # 거리순 정렬 (가까운 것부터)
    nearby_constructions.sort(key=lambda x: x['distance'])
    
    print(f"✅ 분석 완료: {len(nearby_constructions)}건 발견")
    if nearby_constructions:
        print(f"   가장 가까운 공사: {nearby_constructions[0]['distance']:.0f}m")
    
    return nearby_constructions

# =============================================================================
# 3. 개선된 Azure TTS 서비스
# =============================================================================

class SimpleTTSService:
    """개선된 Azure TTS 서비스 - 스피커 직접 출력"""
    
    def __init__(self):
        self.speech_key = os.getenv('AZURE_SPEECH_KEY')
        self.speech_region = os.getenv('AZURE_SPEECH_REGION', 'eastus')
        
        # 기본 음성 설정
        self.voice_name = 'ko-KR-SunHiNeural'  # 한국어 여성 음성
        
        print(f"🔑 Azure Speech Key: {'✅ 설정됨' if self.speech_key else '❌ 없음'}")
        print(f"🌏 Azure Region: {self.speech_region}")
        print(f"🎤 Azure SDK: {'✅ 사용 가능' if AZURE_AVAILABLE else '❌ 설치 필요'}")
        print(f"🔊 출력 방식: 스피커 직접 출력")
    
    def speak_text(self, text):
        """텍스트를 스피커로 직접 출력"""
        # Azure SDK가 없으면 텍스트만 출력
        if not AZURE_AVAILABLE:
            print(f"🔊 [텍스트 출력] {text}")
            return {'success': False, 'text': text, 'reason': 'Azure SDK 없음'}
        
        # API 키가 없으면 텍스트만 출력
        if not self.speech_key:
            print(f"🔊 [텍스트 출력] {text}")
            return {'success': False, 'text': text, 'reason': 'API Key 없음'}
        
        try:
            print(f"🔊 음성 출력 중: {text}")
            
            # Speech Config 설정
            speech_config = speechsdk.SpeechConfig(
                subscription=self.speech_key, 
                region=self.speech_region
            )
            speech_config.speech_synthesis_voice_name = self.voice_name
            
            # 스피커로 직접 출력
            synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config)
            
            # 음성 합성 및 재생
            result = synthesizer.speak_text_async(text).get()
            
            if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
                print(f"✅ 음성 출력 완료")
                return {
                    'success': True,
                    'text': text,
                    'output_method': '스피커 직접 출력'
                }
            else:
                print(f"❌ 음성 합성 실패: {result.reason}")
                return {'success': False, 'text': text, 'reason': f'TTS 실패: {result.reason}'}
                
        except Exception as e:
            print(f"❌ TTS 오류: {str(e)}")
            return {'success': False, 'text': text, 'reason': f'오류: {str(e)}'}

# =============================================================================
# 4. 음성 안내 메시지 생성 (기존과 동일)
# =============================================================================

class VoiceGuidanceGenerator:
    """음성 안내 메시지 생성기"""
    
    def __init__(self, tts_service):
        self.tts_service = tts_service
        
        # 간단한 메시지 템플릿
        self.templates = {
            'no_construction': "주변에 진행중인 공사가 없습니다. 안전한 경로입니다.",
            'immediate': "주의! 전방 {distance}미터에 공사가 진행중입니다. 우회를 권장합니다.",
            'warning': "안내. {distance}미터 앞에 공사 구역이 있습니다.",
            'notice': "참고. 주변 {distance}미터 지점에 공사가 있습니다.",
            'multiple': "주변에 {count}곳의 공사가 진행중입니다. 가장 가까운 곳은 {distance}미터 전방입니다."
        }
    
    def generate_guidance_message(self, constructions_analysis):
        """공사 분석 결과를 바탕으로 안내 메시지 생성"""
        if not constructions_analysis:
            return self.templates['no_construction']
        
        if len(constructions_analysis) == 1:
            construction = constructions_analysis[0]
            urgency = construction['urgency']
            distance = int(construction['distance'])
            
            if urgency in self.templates:
                return self.templates[urgency].format(distance=distance)
        
        else:
            # 여러 공사가 있는 경우
            count = len(constructions_analysis)
            closest_distance = int(constructions_analysis[0]['distance'])
            
            return self.templates['multiple'].format(
                count=count, 
                distance=closest_distance
            )
        
        # 기본 메시지
        return self.templates['notice'].format(
            distance=int(constructions_analysis[0]['distance'])
        )
    
    def create_voice_guidance(self, constructions_analysis):
        """음성 안내 생성 (스피커 직접 출력)"""
        # 메시지 생성
        message = self.generate_guidance_message(constructions_analysis)
        
        # 스피커로 음성 출력
        voice_result = self.tts_service.speak_text(message)
        
        return {
            'message': message,
            'voice_result': voice_result,
            'constructions_count': len(constructions_analysis),
            'constructions_details': constructions_analysis[:3]  # 상위 3개만
        }

# =============================================================================
# 5. 개선된 FR-005 메인 서비스 함수
# =============================================================================

def fr005_voice_guidance_service_improved(user_location, search_radius=500):
    """
    FR-005: 개선된 음성 안내 서비스 (.env 경로 관리 포함)
    
    Args:
        user_location (dict): 사용자 위치 {'lat': 위도, 'lng': 경도}
        search_radius (int): 검색 반경 (미터)
    
    Returns:
        dict: 서비스 결과
    """
    print(f"🚀 FR-005 개선된 음성 안내 서비스 시작")
    print(f"📁 프로젝트 루트: {path_manager.project_root}")
    print(f"📍 사용자 위치: ({user_location['lat']:.6f}, {user_location['lng']:.6f})")
    print(f"🔍 검색 반경: {search_radius}m")
    print("-" * 60)
    
    # 1. 스마트 데이터 로드 (.env 기반)
    construction_data = load_construction_data_smart()
    print(f"📊 로드된 공사 데이터: {len(construction_data)}건")
    
    # 2. 주변 공사 분석
    constructions_analysis = analyze_nearby_constructions(
        user_location, construction_data, search_radius
    )
    
    # 3. TTS 서비스 초기화
    tts_service = SimpleTTSService()
    
    # 4. 음성 안내 생성
    voice_generator = VoiceGuidanceGenerator(tts_service)
    guidance_result = voice_generator.create_voice_guidance(constructions_analysis)
    
    # 5. 상세 결과 출력
    print(f"\n📢 안내 메시지: {guidance_result['message']}")
    
    if guidance_result['constructions_details']:
        print(f"\n📋 발견된 공사 상세 정보:")
        for i, construction in enumerate(guidance_result['constructions_details'], 1):
            print(f"   {i}. 관리코드: {construction['id']}")
            print(f"      위치: {construction['location']}")
            print(f"      거리: {construction['distance']:.0f}m")
            print(f"      위험도: {construction['risk_level']} ({construction['urgency']})")
            print(f"      좌표상태: {construction['coordinate_status']}")
            if construction['end_date'] != '미정':
                if isinstance(construction['end_date'], str):
                    print(f"      완료예정: {construction['end_date']}")
                else:
                    print(f"      완료예정: {construction['end_date'].strftime('%Y-%m-%d')}")
            print()
    else:
        print(f"✅ 주변에 진행중인 공사가 없습니다.")
    
    # 6. 최종 결과 반환
    return {
        'user_location': user_location,
        'search_radius': search_radius,
        'project_root': str(path_manager.project_root),
        'data_source': construction_data.iloc[0]['좌표상태'] if len(construction_data) > 0 else 'none',
        'total_constructions_in_data': len(construction_data),
        'constructions_found': len(constructions_analysis),
        'message': guidance_result['message'],
        'voice_output': guidance_result['voice_result'].get('success', False),
        'constructions_details': guidance_result['constructions_details']
    }

# =============================================================================
# 6. 개선된 테스트 함수들
# =============================================================================

def test_path_management():
    """경로 관리 기능 테스트"""
    print(f"\n📁 경로 관리 테스트")
    print("-" * 50)
    
    print(f"🏠 프로젝트 루트: {path_manager.project_root}")
    
    # 사용 가능한 데이터 파일 확인
    available_files = path_manager.list_available_data_files()
    
    if available_files:
        print(f"\n✅ 경로 관리 성공: {len(available_files)}개 파일 발견")
    else:
        print(f"\n⚠️ 데이터 파일을 찾을 수 없습니다.")
        print(f"💡 다음 중 하나의 파일을 준비하세요:")
        for path in path_manager.get_construction_data_paths():
            print(f"   - {path.relative_to(path_manager.project_root)}")

def test_smart_data_loading():
    """스마트 데이터 로딩 테스트"""
    print(f"\n📊 스마트 데이터 로딩 테스트")
    print("-" * 50)
    
    data = load_construction_data_smart()
    
    if len(data) > 0:
        print(f"✅ 데이터 로드 성공: {len(data)}건")
        print(f"📋 컬럼: {list(data.columns)}")
        
        if '좌표상태' in data.columns:
            print(f"📍 좌표상태 분포:")
            for status, count in data['좌표상태'].value_counts().items():
                print(f"   - {status}: {count}건")
        
        # 샘플 데이터 출력
        print(f"\n📄 샘플 데이터:")
        sample_cols = ['관리코드', '지오코딩주소', '위도', '경도', '공사상태']
        available_cols = [col for col in sample_cols if col in data.columns]
        if available_cols:
            print(data[available_cols].head(3).to_string(index=False))
    else:
        print(f"❌ 데이터 로드 실패")

def test_improved_full_workflow():
    """개선된 전체 워크플로우 테스트"""
    print(f"\n🎬 개선된 전체 워크플로우 테스트")
    print("=" * 60)
    
    # 실제 데이터에 있을 법한 위치들로 테스트
    test_locations = [
        {'name': '잠원동 (실제 공사 가능 지역)', 'lat': 37.5079, 'lng': 127.0035},
        {'name': '강남역 주변', 'lat': 37.4979, 'lng': 127.0276},
        {'name': '서초역 주변', 'lat': 37.4937, 'lng': 127.0134}
    ]
    
    for i, location in enumerate(test_locations, 1):
        print(f"\n📍 테스트 {i}: {location['name']}")
        print(f"   위치: ({location['lat']:.4f}, {location['lng']:.4f})")
        
        try:
            result = fr005_voice_guidance_service_improved({
                'lat': location['lat'], 
                'lng': location['lng']
            }, search_radius=800)
            
            print(f"   📊 결과 요약:")
            print(f"      프로젝트 루트: .../{result['project_root'].split('/')[-1]}")
            print(f"      데이터 소스: {result['data_source']}")
            print(f"      전체 공사: {result['total_constructions_in_data']}건")
            print(f"      발견된 공사: {result['constructions_found']}건")
            print(f"      음성 출력: {'✅' if result['voice_output'] else '❌'}")
            print(f"      메시지: {result['message'][:50]}...")
            
            if result['constructions_details']:
                closest = result['constructions_details'][0]
                print(f"      가장 가까운 공사: {closest['distance']:.0f}m ({closest['risk_level']})")
            
        except Exception as e:
            print(f"   ❌ 테스트 실패: {str(e)}")
        
        print()

def test_env_configuration():
    """.env 설정 테스트"""
    print(f"\n🔧 .env 설정 테스트")
    print("-" * 50)
    
    # 환경 변수 확인
    config_items = [
        ('AZURE_SPEECH_KEY', '🔑 Azure Speech Key'),
        ('AZURE_SPEECH_REGION', '🌏 Azure Region'),
        ('CONSTRUCTION_DATA_PATH', '📂 주 데이터 경로'),
        ('CONSTRUCTION_DATA_FALLBACK_PATHS', '📂 대체 경로들'),
        ('VOICE_OUTPUT_DIR', '🔊 출력 디렉토리')
    ]
    
    for env_key, description in config_items:
        value = os.getenv(env_key)
        if value:
            if env_key == 'AZURE_SPEECH_KEY':
                print(f"   {description}: {'✅ 설정됨' if value and value != 'your_azure_speech_key_here' else '❌ 기본값'}")
            else:
                print(f"   {description}: ✅ {value}")
        else:
            print(f"   {description}: ❌ 설정되지 않음")
    
    # .env 파일 위치 확인
    env_path = path_manager.project_root / '.env'
    print(f"\n📄 .env 파일 위치: {env_path}")
    print(f"   존재 여부: {'✅ 있음' if env_path.exists() else '❌ 없음'}")
    
    if env_path.exists():
        try:
            with open(env_path, 'r', encoding='utf-8') as f:
                content = f.read()
                lines = [line.strip() for line in content.split('\n') if line.strip() and not line.startswith('#')]
                print(f"   설정 항목: {len(lines)}개")
        except Exception as e:
            print(f"   ❌ 파일 읽기 실패: {e}")

def demonstrate_improved_workflow():
    """개선된 워크플로우 시연"""
    print(f"\n🎬 개선된 FR-005 워크플로우 시연")
    print("=" * 70)
    
    print(f"1️⃣ 경로 관리 및 설정 확인...")
    test_path_management()
    
    print(f"\n2️⃣ 환경 변수 설정 확인...")
    test_env_configuration()
    
    print(f"\n3️⃣ 스마트 데이터 로드...")
    data = load_construction_data_smart()
    print(f"   ✅ {len(data)}건의 공사 데이터 준비 완료")
    
    print(f"\n4️⃣ 음성 안내 서비스 실행...")
    user_location = {'lat': 37.5079, 'lng': 127.0035}  # 잠원동
    print(f"   📍 테스트 위치: 잠원동")
    
    result = fr005_voice_guidance_service_improved(user_location, search_radius=600)
    
    print(f"\n5️⃣ 최종 결과 요약:")
    print(f"   🏠 프로젝트 루트: {result['project_root']}")
    print(f"   📊 데이터 소스: {result['data_source']}")
    print(f"   🔍 발견된 공사: {result['constructions_found']}건")
    print(f"   🔊 음성 출력: {'✅ 성공' if result['voice_output'] else '❌ 실패'}")
    print(f"   💬 안내 메시지: {result['message']}")
    
    if result['constructions_details']:
        print(f"   📋 공사 상세:")
        for construction in result['constructions_details']:
            print(f"      - {construction['location'][:40]}... ({construction['distance']:.0f}m)")
    
    print(f"\n✅ 개선된 워크플로우 완료!")
    print(f"   🎯 주요 개선사항:")
    print(f"      - .env 기반 경로 관리")
    print(f"      - 프로젝트 루트 자동 감지")
    print(f"      - 다중 데이터 소스 지원")
    print(f"      - 스마트 파일 검색")
    
    return result

def quick_fix_and_test():
    """빠른 경로 수정 및 테스트"""
    print(f"🔧 빠른 경로 수정 및 테스트")
    print("=" * 50)
    
    # 1. 프로젝트 루트 수정 시도
    print(f"1️⃣ 프로젝트 루트 자동 수정...")
    fixed_manager = fix_project_root()
    
    if fixed_manager:
        print(f"\n2️⃣ 경로 관리 재테스트...")
        test_path_management()
        
        print(f"\n3️⃣ 데이터 로드 테스트...")
        test_smart_data_loading()
        
        print(f"\n✅ 경로 수정 및 테스트 완료!")
        return True
    else:
        print(f"\n❌ 자동 수정 실패")
        print(f"💡 수동 설정 방법:")
        print(f"   # Windows 경로 예시")
        print(f"   correct_path = r'C:\\Users\\User\\Desktop\\5조 팀플\\perror\\Sesac_5_perror'")
        print(f"   path_manager = init_path_manager(correct_path)")
        print(f"   test_path_management()")
        return False

def manual_setup_guide():
    """수동 설정 가이드"""
    print(f"🛠️ 수동 경로 설정 가이드")
    print("=" * 50)
    
    current_detected = path_manager.project_root
    print(f"현재 감지된 루트: {current_detected}")
    
    # Windows 경로 형식 가이드
    print(f"\n📝 Windows 경로 설정 예시:")
    print(f"# 1. 올바른 프로젝트 루트 경로 확인")
    print(f"correct_path = r'C:\\Users\\User\\Desktop\\5조 팀플\\perror\\Sesac_5_perror'")
    print(f"")
    print(f"# 2. 경로 관리자 재설정")
    print(f"path_manager = init_path_manager(correct_path)")
    print(f"")
    print(f"# 3. 설정 확인")
    print(f"test_path_management()")
    
    print(f"\n🔍 현재 경로에서 찾기:")
    current = Path.cwd()
    print(f"작업 디렉토리: {current}")
    
    # 상위 디렉토리들 나열
    print(f"\n📂 상위 디렉토리들:")
    for i, parent in enumerate(current.parents):
        marker = ""
        if 'sesac' in parent.name.lower() or 'perror' in parent.name.lower():
            marker = " ← 이것 같음!"
        print(f"   {i+1}. {parent}{marker}")
        
        if i > 5:  # 너무 많이 올라가지 않도록
            break

# =============================================================================
# 7. 사용 가이드 및 예시
# =============================================================================

print(f"\n📝 FR-005 개선된 TTS 서비스 가이드")
print("=" * 70)
print(f"🎯 주요 개선사항:")
print(f"   ✅ .env 파일 기반 경로 관리")
print(f"   ✅ 프로젝트 루트 자동 감지")
print(f"   ✅ 다중 데이터 소스 지원")
print(f"   ✅ 스마트 파일 검색")
print(f"   ✅ 개선된 데이터 전처리")

print(f"\n📂 폴더 구조:")
print(f"   프로젝트루트/")
print(f"   ├── .env                    # 환경 설정")
print(f"   ├── backend/FR-005/         # 현재 위치")
print(f"   │   └── FR-005-TTS.ipynb")
print(f"   └── data/")
print(f"       ├── processed/          # 전처리된 데이터")
print(f"       └── csv/                # 원본 CSV 파일")

print(f"\n🔧 .env 설정 예시:")
print(f"   AZURE_SPEECH_KEY=your_key_here")
print(f"   AZURE_SPEECH_REGION=eastus")
print(f"   CONSTRUCTION_DATA_PATH=data/processed/road_excavation_processed_20250623_143840.csv")
print(f"   CONSTRUCTION_DATA_FALLBACK_PATHS=data/csv/서울시 도로굴착 공사 현황.csv")

print(f"\n💡 빠른 시작 (경로 문제 해결):")
print(f"   # 1. 경로 자동 수정 시도")
print(f"   quick_fix_and_test()")
print(f"   ")
print(f"   # 2. 수동 설정 가이드")
print(f"   manual_setup_guide()")
print(f"   ")
print(f"   # 3. 수동 경로 설정")
print(f"   correct_path = r'C:\\Users\\User\\Desktop\\5조 팀플\\perror\\Sesac_5_perror'")
print(f"   path_manager = init_path_manager(correct_path)")
print(f"   ")
print(f"   # 4. 설정 확인")
print(f"   test_path_management()")

print(f"\n🎮 실제 서비스 사용:")
print(f"   # 기본 사용법")
print(f"   result = fr005_voice_guidance_service_improved({{'lat': 37.5079, 'lng': 127.0035}})")
print(f"   ")
print(f"   # 검색 반경 조정")
print(f"   result = fr005_voice_guidance_service_improved(")
print(f"       user_location={{'lat': 37.5000, 'lng': 127.0000}},")
print(f"       search_radius=1000")
print(f"   )")

print(f"\n🔊 음성 출력 특징:")
print(f"   - 스피커 직접 출력 (파일 저장 없음)")
print(f"   - Azure TTS 한국어 음성")
print(f"   - 위험도별 메시지 자동 생성")
print(f"   - 실시간 상태 피드백")

print(f"\n⚠️ 문제 해결:")
print(f"   1. 데이터 파일을 찾을 수 없는 경우:")
print(f"      → test_path_management() 실행하여 경로 확인")
print(f"      → .env 파일의 CONSTRUCTION_DATA_PATH 설정")
print(f"   ")
print(f"   2. 음성이 출력되지 않는 경우:")
print(f"      → .env 파일의 AZURE_SPEECH_KEY 확인")
print(f"      → 인터넷 연결 및 스피커 설정 확인")
print(f"   ")
print(f"   3. 공사를 찾을 수 없는 경우:")
print(f"      → search_radius를 늘려서 테스트")
print(f"      → 실제 공사가 있는 위치로 테스트")

print(f"\n🎯 추천 테스트 순서:")
print(f"   1. test_env_configuration()      # 설정 확인")
print(f"   2. test_path_management()        # 경로 확인")
print(f"   3. test_smart_data_loading()     # 데이터 로드")
print(f"   4. demonstrate_improved_workflow() # 전체 테스트")

print(f"\n" + "=" * 70)
print(f"🚀 개선된 FR-005 TTS 서비스 준비 완료!")
print(f"   .env 파일을 설정하고 demonstrate_improved_workflow()를 실행하세요.")

✅ Azure Speech SDK 로드 완료
🔍 현재 경로: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror\backend\FR-005
   검사 중: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror\backend\FR-005
   검사 중: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror\backend
   검사 중: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror
📁 프로젝트 루트 감지 (이름+구조): c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror
✅ .env 파일 로드: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror\.env
🔊 FR-005-SimplifiedTTS: 개선된 음성 안내 서비스
🎯 프로젝트 루트 기준 경로 관리 + 음성 안내

📝 FR-005 개선된 TTS 서비스 가이드
🎯 주요 개선사항:
   ✅ .env 파일 기반 경로 관리
   ✅ 프로젝트 루트 자동 감지
   ✅ 다중 데이터 소스 지원
   ✅ 스마트 파일 검색
   ✅ 개선된 데이터 전처리

📂 폴더 구조:
   프로젝트루트/
   ├── .env                    # 환경 설정
   ├── backend/FR-005/         # 현재 위치
   │   └── FR-005-TTS.ipynb
   └── data/
       ├── processed/          # 전처리된 데이터
       └── csv/                # 원본 CSV 파일

🔧 .env 설정 예시:
   AZURE_SPEECH_KEY=your_key_here
   AZURE_SPEECH_REGION=eastus
   CONSTRUCTION_DATA_PATH=data/processed/road_excavation_proces

In [ ]:
# 전체 워크플로우 시연
result = demonstrate_improved_workflow()


🎬 개선된 FR-005 워크플로우 시연
1️⃣ 경로 관리 및 설정 확인...

📁 경로 관리 테스트
--------------------------------------------------
🏠 프로젝트 루트: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror

📂 사용 가능한 데이터 파일 검색...
   프로젝트 루트: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror
   ✅ data\processed\road_excavation_processed_20250623_143840.csv (82,924 bytes)
   ✅ data\csv\서울시 도로굴착 공사 현황.csv (123,086 bytes)
   ❌ data\csv\서울시 신설공사 추진 현황.csv
   ❌ backend\FR-005\sample_data.csv

✅ 경로 관리 성공: 2개 파일 발견

2️⃣ 환경 변수 설정 확인...

🔧 .env 설정 테스트
--------------------------------------------------
   🔑 Azure Speech Key: ✅ 설정됨
   🌏 Azure Region: ✅ eastus
   📂 주 데이터 경로: ✅ data/processed/road_excavation_processed_20250623_143840.csv
   📂 대체 경로들: ✅ data/csv/서울시 도로굴착 공사 현황.csv,data/csv/서울시 신설공사 추진 현황.csv
   🔊 출력 디렉토리: ✅ output/voice_guidance

📄 .env 파일 위치: c:\Users\User\Desktop\5조 팀플\perror\Sesac_5_perror\.env
   존재 여부: ✅ 있음
   설정 항목: 20개

3️⃣ 스마트 데이터 로드...
🔍 스마트 데이터 로드 시작...

📂 사용 가능한 데이터 파일 검색...
   프로젝트 루트: c:\Users\User\Desktop\5